# YOLOv8 PPE Safety Detection — Colab Training Notebook

**Goal:** Fine-tune keremberke YOLOv8m-protective-equipment on merged PPE dataset (~19K images)

**Timeline:** ~3-4 hours total
- Setup: 10 min
- Dataset download: 20-30 min
- Merge: 5 min
- Training (30 epochs): 2-3 hours
- Export: 5 min

**Requirements:**
- Google account (Colab free tier OK)
- Roboflow API key (free at https://roboflow.com) — OPTIONAL
- Kaggle credentials for SH17 — OPTIONAL
- Google Drive mounted for checkpoints

## [1] Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
import os
os.makedirs('/content/workspace', exist_ok=True)
os.chdir('/content/workspace')

print("✅ Google Drive mounted at /content/drive")
print("✅ Working directory: /content/workspace")

## [2] Install Dependencies

In [ ]:
!pip install -q ultralytics torch torchvision opencv-python pyyaml albumentations roboflow kaggle huggingface-hub

print("✅ All dependencies installed")

## [3] Clone Repository & Setup Project Structure

In [ ]:
# Clone the project repository
!git clone -q https://github.com/A-Kuo/Yolov8-Powered-Safety-Equipment-Detection-System.git project
!cd project && git checkout claude/setup-yolo-safety-detection-UoZTY

# Setup symlinks for easy access
import shutil
os.chdir('/content/workspace')

# Copy preprocessing scripts
for script in ['download_datasets.py', 'merge_datasets.py', 'augmentation.py']:
    src = f'/content/workspace/project/data/preprocessing/{script}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/workspace/{script}')

# Copy config files
shutil.copytree('/content/workspace/project/data/annotations', '/content/workspace/annotations', dirs_exist_ok=True)
shutil.copytree('/content/workspace/project/config', '/content/workspace/config', dirs_exist_ok=True)

print("✅ Project cloned and scripts ready")
print(f"📁 Working directory: {os.getcwd()}")

## [4] Download Ultralytics Construction-PPE (Zero Setup)

In [ ]:
# Trigger Ultralytics auto-download
from ultralytics import YOLO
import subprocess

print("Downloading Construction-PPE dataset...")
result = subprocess.run(
    ['yolo', 'detect', 'train', 'data=construction-ppe.yaml', 'epochs=0', 'exist_ok=True'],
    capture_output=True,
    timeout=300
)

# Find dataset location
import glob
dataset_paths = glob.glob(os.path.expanduser('~/.cache/ultralytics/datasets/construction-ppe'), recursive=False)
if dataset_paths:
    ULTRALYTICS_DATASET = dataset_paths[0]
else:
    ULTRALYTICS_DATASET = os.path.expanduser('~/ultralytics/datasets/construction-ppe')

print(f"✅ Construction-PPE dataset: {ULTRALYTICS_DATASET}")
!ls -lh {ULTRALYTICS_DATASET}/images/train | head -5

## [5] Download Roboflow Datasets (Optional — Requires API Key)

In [ ]:
# ⚠️ OPTIONAL: Only if you have Roboflow API key
# Set your API key here or via environment variable

ROBOFLOW_API_KEY = None  # Set to your key or leave None to skip

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    import os
    
    os.environ['ROBOFLOW_API_KEY'] = ROBOFLOW_API_KEY
    
    # Download Construction Safety dataset
    print("Downloading Roboflow Construction Site Safety...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
    dataset1 = project.version(27).download("yolov8", location='/content/workspace/roboflow_construction')
    print(f"✅ Construction Safety: {dataset1.location}")
    
    # Download Hard Hat Universe
    print("\nDownloading Hard Hat Universe...")
    project2 = rf.workspace("universe-datasets").project("hard-hat-universe-0dy7t")
    dataset2 = project2.version(1).download("yolov8", location='/content/workspace/hard_hat_universe')
    print(f"✅ Hard Hat Universe: {dataset2.location}")
else:
    print("⏭️  Roboflow datasets skipped (no API key provided)")
    print("   To enable: Set ROBOFLOW_API_KEY above and re-run this cell")
    print("   Get free key at: https://app.roboflow.com/settings/api")

## [6] Download SH17 Dataset (Optional — Requires Kaggle)

In [ ]:
# ⚠️ OPTIONAL: Only if Kaggle CLI is configured
# Place your kaggle.json at ~/.kaggle/kaggle.json

try:
    print("Attempting to download SH17 from Kaggle...")
    !kaggle datasets download mugheesahmad/sh17-dataset-for-ppe-detection -p /content/workspace --unzip
    print("✅ SH17 dataset downloaded")
except Exception as e:
    print(f"⏭️  SH17 skipped: {e}")
    print("   To enable: Upload ~/.kaggle/kaggle.json to Colab")

## [7] Organize Datasets for Merging

In [ ]:
import os
import shutil

# Create data/raw structure
os.makedirs('/content/workspace/data/raw', exist_ok=True)

# Copy Ultralytics dataset
if os.path.exists(ULTRALYTICS_DATASET):
    dest = '/content/workspace/data/raw/ultralytics_construction_ppe'
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(ULTRALYTICS_DATASET, dest)
    print(f"✅ Copied Ultralytics dataset")

# Move Roboflow datasets if they exist
if os.path.exists('/content/workspace/roboflow_construction'):
    shutil.move('/content/workspace/roboflow_construction', '/content/workspace/data/raw/')
    print(f"✅ Moved Roboflow Construction Safety")

if os.path.exists('/content/workspace/hard_hat_universe'):
    shutil.move('/content/workspace/hard_hat_universe', '/content/workspace/data/raw/')
    print(f"✅ Moved Hard Hat Universe")

# Move SH17 if it exists
if os.path.exists('/content/workspace/SH17dataset'):
    shutil.move('/content/workspace/SH17dataset', '/content/workspace/data/raw/sh17')
    print(f"✅ Moved SH17 dataset")

print("\n📁 Raw datasets organized:")
!ls -lh /content/workspace/data/raw/

## [8] Merge All Datasets Using Class Mapping

In [ ]:
# Copy merge script and class mapping
import shutil
shutil.copy('/content/workspace/project/data/preprocessing/merge_datasets.py', '/content/workspace/')
shutil.copy('/content/workspace/project/data/annotations/class_mapping.yaml', '/content/workspace/annotations/')
shutil.copy('/content/workspace/project/data/annotations/classes.yaml', '/content/workspace/annotations/')

# Setup data directory structure
os.makedirs('/content/workspace/data/annotations', exist_ok=True)
shutil.copytree('/content/workspace/annotations', '/content/workspace/data/annotations', dirs_exist_ok=True)
shutil.copytree('/content/workspace/config', '/content/workspace/data/config', dirs_exist_ok=True)

print("Running dataset merge...")
os.chdir('/content/workspace')

import yaml
import logging
logging.basicConfig(level=logging.INFO)

# Execute merge
!python merge_datasets.py

print("\n✅ Merge complete!")
!ls -lh /content/workspace/data/processed/merged/ || echo "Merged dataset directory created"

## [9] Download Pretrained Model & Prepare Training

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

print("Downloading keremberke PPE pretrained model...")
model_path = hf_hub_download(
    repo_id="keremberke/yolov8m-protective-equipment-detection",
    filename="best.pt",
)

# Copy to models directory
os.makedirs('/content/workspace/models', exist_ok=True)
import shutil
shutil.copy(model_path, '/content/workspace/models/keremberke_ppe_m.pt')

print(f"✅ Pretrained model ready: /content/workspace/models/keremberke_ppe_m.pt")

# Check dataset structure
print("\nMerged dataset structure:")
!ls -lh /content/workspace/data/processed/merged/train/images | head -3
!echo "..."
!ls -lh /content/workspace/data/processed/merged/train/labels | head -3

## [10] Create Training Dataset YAML

In [ ]:
import yaml

# Create dataset.yaml for training
dataset_yaml = {
    'path': '/content/workspace/data/processed/merged',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 10,
    'names': {
        0: 'worker',
        1: 'drone',
        2: 'safety_glasses',
        3: 'safety_goggles',
        4: 'hard_hat',
        5: 'regular_hat',
        6: 'hi_vis_vest',
        7: 'regular_clothing',
        8: 'work_boots',
        9: 'regular_shoes'
    }
}

with open('/content/workspace/data/merged_dataset.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print("✅ Training dataset YAML created")
print(yaml.dump(dataset_yaml))

## [11] Start Fine-Tuning Training

In [ ]:
from ultralytics import YOLO
import torch

# Check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "CPU")

# Load pretrained model
model = YOLO('/content/workspace/models/keremberke_ppe_m.pt')

# Fine-tune training
print("\n🚀 Starting fine-tuning...")
results = model.train(
    data='/content/workspace/data/merged_dataset.yaml',
    epochs=30,  # Start with 30, can increase
    imgsz=640,
    batch=16,  # Reduce to 8 if OOM
    device=0,  # GPU
    patience=10,  # Early stopping
    save=True,
    project='/content/workspace/runs',
    name='ppe_finetune_v1',
    exist_ok=True,
    # Optimizer settings for fine-tuning
    optimizer='SGD',
    lr0=0.001,  # Lower LR for fine-tuning
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    amp=True,  # Automatic mixed precision (FP16)
    # Freeze backbone for transfer learning
    freeze=10,
    # Augmentation
    hsv_h=0.010,
    hsv_s=0.50,
    hsv_v=0.40,
    degrees=15,
    translate=0.1,
    scale=0.50,
    flipud=0.0,
    fliplr=0.50,
    mosaic=1.0,
    mixup=0.1,
)

print("\n✅ Training complete!")

## [12] Evaluate Model Performance

In [ ]:
# Load best model
best_model = YOLO('/content/workspace/runs/ppe_finetune_v1/weights/best.pt')

# Validate on test set
print("Validating on test set...")
metrics = best_model.val(
    data='/content/workspace/data/merged_dataset.yaml',
    imgsz=640,
    batch=32,
    device=0,
)

print("\n📊 Validation Results:")
print(f"mAP@50: {metrics.box.map50:.3f}")
print(f"mAP@50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall: {metrics.box.mr:.3f}")

## [13] Test on Sample Image

In [ ]:
# Get a sample image from test set
import glob
import cv2
from matplotlib import pyplot as plt

test_images = glob.glob('/content/workspace/data/processed/merged/test/images/*.jpg')
test_images += glob.glob('/content/workspace/data/processed/merged/test/images/*.png')

if test_images:
    # Run inference on first few test images
    print(f"Testing on {min(3, len(test_images))} sample images...")
    results = best_model.predict(test_images[:3], conf=0.5, imgsz=640)
    
    # Display results
    fig, axes = plt.subplots(1, min(3, len(results)), figsize=(15, 5))
    for idx, result in enumerate(results[:3]):
        img = result.plot()
        if min(3, len(results)) == 1:
            axes.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/workspace/inference_samples.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print("✅ Inference test complete")
else:
    print("⚠️  No test images found")

## [14] Export to ONNX & Prepare for Edge Deployment

In [ ]:
# Export to ONNX
print("Exporting to ONNX format...")
onnx_path = best_model.export(format='onnx', imgsz=640, opset=12)
print(f"✅ ONNX model: {onnx_path}")

# Export to TorchScript (for fallback)
print("\nExporting to TorchScript format...")
ts_path = best_model.export(format='torchscript', imgsz=640)
print(f"✅ TorchScript model: {ts_path}")

# Copy exports to models directory
import shutil
os.makedirs('/content/workspace/models/exports', exist_ok=True)
shutil.copy(onnx_path, '/content/workspace/models/exports/ppe_detector.onnx')
shutil.copy(ts_path, '/content/workspace/models/exports/ppe_detector.torchscript')
shutil.copy('/content/workspace/runs/ppe_finetune_v1/weights/best.pt', '/content/workspace/models/exports/ppe_detector.pt')

print("\n✅ All model formats ready for deployment")
!ls -lh /content/workspace/models/exports/

## [15] Save Results to Google Drive

In [ ]:
import shutil
import os

drive_output = '/content/drive/MyDrive/YOLOv8_PPE_Training'
os.makedirs(drive_output, exist_ok=True)

# Copy trained models
print("Saving to Google Drive...")
shutil.copytree(
    '/content/workspace/models/exports',
    f'{drive_output}/models',
    dirs_exist_ok=True
)

# Copy training runs
shutil.copytree(
    '/content/workspace/runs/ppe_finetune_v1',
    f'{drive_output}/training_results',
    dirs_exist_ok=True
)

# Copy inference samples
if os.path.exists('/content/workspace/inference_samples.png'):
    shutil.copy('/content/workspace/inference_samples.png', f'{drive_output}/')

# Create summary
summary = f"""YOLOv8 PPE Detection Training Summary
=====================================

Models Available:
- ppe_detector.pt (PyTorch)
- ppe_detector.onnx (ONNX Runtime)
- ppe_detector.torchscript (TorchScript)

Performance Metrics:
- mAP@50: {metrics.box.map50:.3f}
- mAP@50-95: {metrics.box.map:.3f}
- Precision: {metrics.box.mp:.3f}
- Recall: {metrics.box.mr:.3f}

Classes (10):
0: worker, 1: drone, 2: safety_glasses, 3: safety_goggles,
4: hard_hat, 5: regular_hat, 6: hi_vis_vest, 7: regular_clothing,
8: work_boots, 9: regular_shoes

Next Steps:
1. Download models folder
2. Test on local warehouse footage
3. Deploy to Snapdragon via ONNX or QNN
4. Iterate on accuracy/performance
"""

with open(f'{drive_output}/TRAINING_SUMMARY.txt', 'w') as f:
    f.write(summary)

print(f"✅ Results saved to: {drive_output}")
print("\n📥 Download folder from Google Drive and use these models for deployment")

## [16] Generate Deployment Instructions

In [ ]:
deployment_guide = """# PPE Detection Model Deployment Guide

## Quick Start

### Option 1: PyTorch (Local Testing)
```python
from ultralytics import YOLO
import cv2

model = YOLO('ppe_detector.pt')
results = model.predict('warehouse_video.mp4', conf=0.5)
```

### Option 2: ONNX Runtime (Cross-Platform)
```python
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('ppe_detector.onnx', providers=['CPUExecutionProvider'])
# Run inference...
```

### Option 3: Snapdragon Edge (QNN)
1. Convert ONNX to Qualcomm DLC:
   ```bash
   # Use Qualcomm AI Hub or QNN SDK
   qnn-convert --input ppe_detector.onnx --output ppe_detector.dlc
   ```

2. Run on Rubik Pi / Snapdragon:
   ```python
   from src.edge_deployment.qnn_executor import QNNExecutor
   executor = QNNExecutor('ppe_detector.dlc')
   results = executor.infer(frame)
   ```

## Classes
- 0: worker (person)
- 1: drone (filter false positives)
- 2-3: eye protection (safety_glasses, safety_goggles)
- 4-5: head protection (hard_hat, regular_hat)
- 6-7: visibility (hi_vis_vest, regular_clothing)
- 8-9: foot protection (work_boots, regular_shoes)

## Compliance Check Logic
```python
def is_compliant(detections):
    has_worker = any(d.class_id == 0 for d in detections)
    has_helmet = any(d.class_id == 4 for d in detections)
    has_vest = any(d.class_id == 6 for d in detections)
    has_eye_protection = any(d.class_id in [2, 3] for d in detections)
    has_foot_protection = any(d.class_id in [8] for d in detections)
    
    return has_worker and has_helmet and has_vest
```

## Performance Targets
- FPS: 30 on Snapdragon (15+ on CPU)
- Latency: <33ms per frame
- Memory: <2GB on Snapdragon
- Accuracy: 95%+ worker detection, 90%+ PPE

## Troubleshooting
- Low FPS? → Reduce image size (640→480)
- Low accuracy? → Increase confidence threshold
- Missing detections? → Fine-tune on warehouse-specific data
"""

with open(f'{drive_output}/DEPLOYMENT_GUIDE.md', 'w') as f:
    f.write(deployment_guide)

print("✅ Deployment guide created")
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print(f"\n📁 Results saved to: {drive_output}")
print("\n✅ Ready for edge deployment")
print("\nNext Steps:")
print("1. Download models from Google Drive")
print("2. Test on local warehouse video")
print("3. Deploy to Snapdragon device")
print("4. Monitor accuracy and optimize")